# 04 — What did the first model learn?

Logistic Regression is additive in TF-IDF feature space. A positive coefficient raises a
class score; a negative coefficient lowers it. Coefficients reveal association with weak
publication labels, **not causal evidence or a universal bias dictionary**.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

DATA_PATH = ROOT / "data/processed/publication_proxy_headlines.csv"

In [2]:
import joblib
import pandas as pd

from bias_dataset.modeling import (
    LABEL_ORDER, attach_split, coefficient_table, explain_document, load_modeling_data
)

SPLIT_PATH = ROOT / "data/splits/pilot_v1.csv"
MODEL_PATH = ROOT / "models/selected_development_lr.joblib"
REPORT_DIR = ROOT / "reports/modeling"

## Load the selected development model

In [3]:
model = joblib.load(MODEL_PATH)
vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]

print("N-gram range:", vectorizer.ngram_range)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("Classes learned:", list(classifier.classes_))

N-gram range: (1, 1)
Vocabulary size: 659
Classes learned: ['Center', 'Lean Left', 'Lean Right', 'Left', 'Right']


## Build the complete coefficient table

In [4]:
coefficients = coefficient_table(model)
coefficients.head()

,class,feature,coefficient
0,Center,000,-0.233337
1,Center,11,-0.171398
2,Center,13,-0.108282
3,Center,200k,0.385386
4,Center,2028,0.027581


## Top positive associations for each class

In [5]:
top_positive = (
    coefficients.sort_values(["class", "coefficient"], ascending=[True, False])
    .groupby("class", sort=False).head(15)
)
top_positive.pivot(index="feature", columns="class", values="coefficient").fillna("")

class,Center,Lean Left,Lean Right,Left,Right
feature,,,,,
abbey,,,,,0.652269
accused,,,,,0.632932
administration,,,1.261442,,
after,,,,,0.753725
amendment,,,0.6424,,
...,...,...,...,...,...
ukraine,,,,0.997801,
us,1.36613,,,,
voters,,,,,0.809018


Read these as: “holding other observed features fixed, this phrase pushes the linear score
toward this outlet class.” Topic, named-entity, and publication fingerprints can all appear.

## Most negative associations for each class

In [6]:
top_negative = (
    coefficients.sort_values(["class", "coefficient"], ascending=[True, True])
    .groupby("class", sort=False).head(10)
)
top_negative.pivot(index="feature", columns="class", values="coefficient").fillna("")

class,Center,Lean Left,Lean Right,Left,Right
feature,,,,,
administration,,-0.459,,-0.64797,
after,,,-0.678016,,
american,-0.514121,-0.502933,,,
and,-0.566906,,,-0.783312,
are,-0.52375,-0.462021,,,
as,,,,-0.62131,
at,,,,,-0.518536
ballroom,,-0.397051,,,
by,,,,,-0.534179


## Audit obvious publisher-fingerprint terms

In [7]:
fingerprint_terms = {
    "cnn", "fox", "newsmax", "breitbart", "mediaite", "vox", "reason",
    "mother jones", "daily wire", "the hill", "newsnation", "dispatch",
    "examiner", "atlantic", "abc", "nbc",
}
fingerprint_hits = coefficients[coefficients["feature"].isin(fingerprint_terms)]
fingerprint_hits.reindex(fingerprint_hits["coefficient"].abs().sort_values(ascending=False).index).head(30)

,class,feature,coefficient
894,Lean Left,fox,0.631029
781,Lean Left,cnn,0.444591
122,Center,cnn,-0.203811
235,Center,fox,-0.176913
2871,Right,fox,-0.171760
1440,Lean Right,cnn,-0.160196
1553,Lean Right,fox,-0.153070
2099,Left,cnn,-0.150930
2212,Left,fox,-0.129286
2758,Right,cnn,0.070345


A fingerprint hit is a warning, not automatic proof of leakage: an outlet name can be the
subject of a legitimate political headline. Large branded coefficients should nevertheless
motivate later masking and source-specific error audits.

## Choose a correctly predicted validation example

In [8]:
data = load_modeling_data(DATA_PATH)
manifest = pd.read_csv(SPLIT_PATH)
validation = attach_split(data, manifest).query("split == 'validation'").copy()

probabilities = model.predict_proba(validation["model_text"])
validation["prediction"] = model.classes_[probabilities.argmax(axis=1)]
validation["confidence"] = probabilities.max(axis=1)
example = validation[validation["prediction"].eq(validation["weak_label"])].nlargest(1, "confidence").iloc[0]
example[["headline", "weak_label", "prediction", "confidence"]]

headline      Today in Supreme Court History: August 24, 1946
weak_label                                         Lean Right
prediction                                         Lean Right
confidence                                           0.806783
Name: 443, dtype: object

## Decompose that prediction feature by feature

In [9]:
explanation = explain_document(model, example["model_text"], example["prediction"])
explanation.head(15)

,feature,tfidf,coefficient,contribution
0,august,0.467465,1.112646,0.520123
1,today,0.467465,1.112646,0.520123
2,history,0.460137,1.053656,0.484827
3,court,0.355960,1.278871,0.455227
4,supreme,0.397707,0.849751,0.337951
5,in,0.257581,-0.043246,-0.011139


`contribution = TF-IDF value × class coefficient`. The intercept and the competing classes'
scores also affect the final softmax probability, so the table explains the chosen class
score rather than claiming each token independently caused the prediction.

## Save compact interpretation artifacts

In [10]:
top_features = pd.concat([
    top_positive.assign(direction="positive"),
    top_negative.assign(direction="negative"),
])
top_features.to_csv(REPORT_DIR / "top_model_features.csv", index=False)
explanation.to_csv(REPORT_DIR / "example_feature_contributions.csv", index=False)
print("Saved top features and the example decomposition.")

Saved top features and the example decomposition.
